In [ ]:
import pandas as pd

from project_paths import project_root as get_project_root
from pipeline_config import ELO_FAVORITE_THRESHOLD, classify_elo_group

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 20)

base_path = get_project_root()
bronze_path = base_path / 'data' / 'bronze'
silver_path = base_path / 'data' / 'silver'
outputs_tables_path = base_path / 'outputs' / 'tables'

team_match_metrics_path = bronze_path / 'team_match_statsbomb_metrics.parquet'
bronze_output_path = bronze_path / 'elo_ratings.parquet'
elo_features_output_path = silver_path / 'team_match_elo_features.parquet'
missing_elo_output_path = outputs_tables_path / 'bd08_missing_elo_values.csv'

for path in [silver_path, outputs_tables_path]:
    path.mkdir(parents=True, exist_ok=True)

{
    'team_match_metrics_input': 'data/bronze/team_match_statsbomb_metrics.parquet',
    'elo_input': 'data/bronze/elo_ratings.parquet',
    'elo_features_output': 'data/silver/team_match_elo_features.parquet',
    'elo_threshold': ELO_FAVORITE_THRESHOLD,
}


# 02b Elo-Differenz und Elo-Gruppen

Dieser Abschnitt nutzt die in BD-06 erzeugten Team-Match-Metriken und die in BD-07 bereinigten historischen Elo-Ratings. Fuer jede Team-Match-Zeile wird per As-of-Join der letzte bekannte Elo-Wert am oder vor dem Spieltag gesucht. Danach werden Gegner-Elo, `elo_diff` und die fachlichen Gruppen `favorite`, `balanced` und `underdog` berechnet.

Output:

- `data/silver/team_match_elo_features.parquet`
- `outputs/tables/bd08_missing_elo_values.csv`

### Methodischer Ansatz

Die Team-Match-Tabelle wird mit historischen Elo-Werten verbunden. Da Elo-Ratings Stichtagsdaten sind, wird je Team der letzte Rating-Datensatz am oder vor dem Matchdatum verwendet. Fehlende Join-Ergebnisse werden dokumentiert, damit die Datenqualitaet nachvollziehbar bleibt.

In [ ]:
team_matches = pd.read_parquet(team_match_metrics_path)
elo_history = pd.read_parquet(bronze_output_path)

team_matches['match_date'] = pd.to_datetime(team_matches['match_date'])
elo_history['match_date'] = pd.to_datetime(elo_history['match_date'])

{
    'team_match_rows': len(team_matches),
    'unique_matches': int(team_matches['match_id'].nunique()),
    'elo_rows': len(elo_history),
    'team_match_metrics_input': 'data/bronze/team_match_statsbomb_metrics.parquet',
    'elo_input': 'data/bronze/elo_ratings.parquet',
    'elo_features_output': 'data/silver/team_match_elo_features.parquet',
}

### Letzten Elo-Wert vor dem Spiel finden

Die Funktion unten fuehrt den As-of-Join gruppiert nach Teamnamen aus. Dadurch bleibt die Team-Match-Tabelle vollstaendig erhalten; falls fuer ein Team kein historischer Elo-Wert gefunden wird, bleibt der Rating-Wert leer und wird dokumentiert.

In [ ]:
def attach_latest_elo(left: pd.DataFrame, history: pd.DataFrame, team_column: str, prefix: str) -> pd.DataFrame:
    rating_columns = [
        'team_name',
        'match_date',
        'elo_rating',
        'elo_team_name',
        'elo_team_code',
        'source_page',
    ]
    history_prepared = (
        history[rating_columns]
        .dropna(subset=['team_name', 'match_date', 'elo_rating'])
        .sort_values(['team_name', 'match_date'])
        .rename(columns={
            'team_name': '_elo_join_team_name',
            'match_date': f'{prefix}_elo_match_date',
            'elo_rating': f'{prefix}_elo',
            'elo_team_name': f'{prefix}_elo_team_name',
            'elo_team_code': f'{prefix}_elo_team_code',
            'source_page': f'{prefix}_elo_source_page',
        })
    )

    merged_parts = []
    left_prepared = left.copy()
    left_prepared['_row_id'] = range(len(left_prepared))

    for team_name, team_rows in left_prepared.groupby(team_column, dropna=False):
        team_rows = team_rows.sort_values(['match_date', 'match_id', '_row_id'])
        team_history = history_prepared[history_prepared['_elo_join_team_name'] == team_name]

        if team_history.empty:
            empty = team_rows.copy()
            empty[f'{prefix}_elo_match_date'] = pd.NaT
            empty[f'{prefix}_elo'] = pd.NA
            empty[f'{prefix}_elo_team_name'] = pd.NA
            empty[f'{prefix}_elo_team_code'] = pd.NA
            empty[f'{prefix}_elo_source_page'] = pd.NA
            merged_parts.append(empty)
            continue

        merged = pd.merge_asof(
            team_rows,
            team_history.sort_values(f'{prefix}_elo_match_date'),
            left_on='match_date',
            right_on=f'{prefix}_elo_match_date',
            direction='backward',
            allow_exact_matches=True,
        ).drop(columns=['_elo_join_team_name'])
        merged_parts.append(merged)

    result = pd.concat(merged_parts, ignore_index=True).sort_values('_row_id')
    return result.drop(columns=['_row_id']).reset_index(drop=True)

team_with_elo = attach_latest_elo(team_matches, elo_history, 'team_name', 'team')
team_with_both_elos = attach_latest_elo(team_with_elo, elo_history, 'opponent_team_name', 'opponent')

team_with_both_elos[[
    'match_id', 'match_date', 'team_name', 'opponent_team_name',
    'team_elo', 'team_elo_match_date', 'opponent_elo', 'opponent_elo_match_date'
]].head(10)

### Elo-Differenz und Gruppen berechnen

Gruppenlogik fuer BD-08:

- `favorite`, wenn `elo_diff > 75`
- `balanced`, wenn `-75 <= elo_diff <= 75`
- `underdog`, wenn `elo_diff < -75`

In [ ]:
elo_features = team_with_both_elos.copy()
elo_features['team_elo'] = pd.to_numeric(elo_features['team_elo'], errors='coerce')
elo_features['opponent_elo'] = pd.to_numeric(elo_features['opponent_elo'], errors='coerce')
elo_features['elo_diff'] = elo_features['team_elo'] - elo_features['opponent_elo']
elo_features['elo_group'] = elo_features['elo_diff'].map(classify_elo_group)

team_missing = elo_features['team_elo'].isna()
opponent_missing = elo_features['opponent_elo'].isna()
elo_features['elo_missing_reason'] = pd.NA
elo_features.loc[team_missing & opponent_missing, 'elo_missing_reason'] = 'team_and_opponent_elo_missing'
elo_features.loc[team_missing & ~opponent_missing, 'elo_missing_reason'] = 'team_elo_missing'
elo_features.loc[~team_missing & opponent_missing, 'elo_missing_reason'] = 'opponent_elo_missing'

feature_columns = [
    'match_id',
    'scope_id',
    'short_name',
    'competition_id',
    'season_id',
    'competition_name',
    'statsbomb_competition_name',
    'season_name',
    'match_date',
    'kick_off',
    'match_week',
    'competition_stage_name',
    'stadium_name',
    'stadium_country_name',
    'team_id',
    'team_name',
    'opponent_team_id',
    'opponent_team_name',
    'is_home',
    'team_score',
    'opponent_score',
    'xg',
    'shots',
    'xg_per_shot',
    'passes',
    'successful_passes',
    'pass_completion_rate',
    'pressures',
    'counterpressures',
    'duels',
    'carries',
    'long_balls',
    'long_ball_share',
    'team_elo',
    'team_elo_match_date',
    'team_elo_team_name',
    'team_elo_team_code',
    'team_elo_source_page',
    'opponent_elo',
    'opponent_elo_match_date',
    'opponent_elo_team_name',
    'opponent_elo_team_code',
    'opponent_elo_source_page',
    'elo_diff',
    'elo_group',
    'elo_missing_reason',
]

elo_features = elo_features[feature_columns].sort_values(['match_id', 'is_home'], ascending=[True, False])
elo_features.to_parquet(elo_features_output_path, index=False)

missing_elo_values = elo_features[elo_features['elo_missing_reason'].notna()].copy()
missing_elo_values[[
    'match_id', 'match_date', 'team_name', 'opponent_team_name',
    'team_elo', 'opponent_elo', 'elo_missing_reason'
]].to_csv(missing_elo_output_path, index=False)

{
    'rows': len(elo_features),
    'matches': int(elo_features['match_id'].nunique()),
    'missing_elo_rows': len(missing_elo_values),
    'elo_group_counts': elo_features['elo_group'].value_counts(dropna=False).to_dict(),
    'missing_elo_output': 'outputs/tables/bd08_missing_elo_values.csv',
}

### Qualitaetschecks

Die Checks bilden die BD-08-Akzeptanzkriterien ab: Jede Team-Match-Zeile besitzt Team-Elo, Gegner-Elo und Elo-Differenz; die Gruppe entspricht den Schwellenwerten; fehlende Werte sind als CSV dokumentiert.

In [ ]:
required_elo_columns = ['team_elo', 'opponent_elo', 'elo_diff', 'elo_group']
missing_required_elo_values = elo_features[required_elo_columns].isna().sum().to_dict()

wrong_favorite_rows = elo_features[(elo_features['elo_group'] == 'favorite') & ~(elo_features['elo_diff'] > ELO_FAVORITE_THRESHOLD)]
wrong_balanced_rows = elo_features[(elo_features['elo_group'] == 'balanced') & ~elo_features['elo_diff'].between(-ELO_FAVORITE_THRESHOLD, ELO_FAVORITE_THRESHOLD, inclusive='both')]
wrong_underdog_rows = elo_features[(elo_features['elo_group'] == 'underdog') & ~(elo_features['elo_diff'] < -ELO_FAVORITE_THRESHOLD)]

quality_checks_bd08 = {
    'output_exists': elo_features_output_path.exists(),
    'missing_elo_documentation_exists': missing_elo_output_path.exists(),
    'input_team_match_rows': len(team_matches),
    'output_rows': len(elo_features),
    'missing_required_elo_values': missing_required_elo_values,
    'wrong_favorite_rows': len(wrong_favorite_rows),
    'wrong_balanced_rows': len(wrong_balanced_rows),
    'wrong_underdog_rows': len(wrong_underdog_rows),
    'teams_without_team_elo': sorted(elo_features.loc[elo_features['team_elo'].isna(), 'team_name'].dropna().unique().tolist()),
    'teams_without_opponent_elo': sorted(elo_features.loc[elo_features['opponent_elo'].isna(), 'opponent_team_name'].dropna().unique().tolist()),
}

assert quality_checks_bd08['output_exists']
assert quality_checks_bd08['missing_elo_documentation_exists']
assert quality_checks_bd08['output_rows'] == quality_checks_bd08['input_team_match_rows']
assert all(value == 0 for value in missing_required_elo_values.values())
assert quality_checks_bd08['wrong_favorite_rows'] == 0
assert quality_checks_bd08['wrong_balanced_rows'] == 0
assert quality_checks_bd08['wrong_underdog_rows'] == 0

quality_checks_bd08

In [ ]:
elo_features[[
    'match_id',
    'team_name',
    'opponent_team_name',
    'team_elo',
    'opponent_elo',
    'elo_diff',
    'elo_group',
]].head(20)